In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/arhamrumi/amazon-product-reviews/Reviews.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')
 
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
 
print("All imports successful.")

All imports successful.


In [3]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/arhamrumi/amazon-product-reviews/Reviews.csv')
print(df.shape)
print(df.columns.tolist())
df.head(3)

(568454, 10)
['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


In [4]:
print(df['Score'].value_counts().sort_index())
print(f"\nNulls:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum()}")

Score
1     52268
2     29769
3     42640
4     80655
5    363122
Name: count, dtype: int64

Nulls:
Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

Duplicates: 0


In [5]:
# Keep only relevant columns
df = df[['UserId', 'ProductId', 'Score', 'Time', 'Summary', 'Text']].copy()

# Drop the 27 rows with nulls in Summary (tiny loss, clean data)
df = df.dropna()

# Convert timestamp
df['Time'] = pd.to_datetime(df['Time'], unit='s')

print(f"Shape after cleaning: {df.shape}")
print(f"\nDate range: {df['Time'].min()} → {df['Time'].max()}")
print(f"\nUnique users   : {df['UserId'].nunique():,}")
print(f"Unique products: {df['ProductId'].nunique():,}")

Shape after cleaning: (568427, 6)

Date range: 1999-10-08 00:00:00 → 2012-10-26 00:00:00

Unique users   : 256,056
Unique products: 74,258


In [6]:
n_users = df['UserId'].nunique()
n_products = df['ProductId'].nunique()
n_interactions = len(df)

sparsity = 1 - (n_interactions / (n_users * n_products))

print(f"Users              : {n_users:,}")
print(f"Products           : {n_products:,}")
print(f"Interactions       : {n_interactions:,}")
print(f"Possible pairs     : {n_users * n_products:,}")
print(f"Sparsity           : {sparsity * 100:.4f}%")

Users              : 256,056
Products           : 74,258
Interactions       : 568,427
Possible pairs     : 19,014,206,448
Sparsity           : 99.9970%


In [7]:
user_counts = df.groupby('UserId').size()

print(f"Users with exactly 1 review : {(user_counts == 1).sum():,} ({(user_counts == 1).mean()*100:.1f}%)")
print(f"Users with < 5 reviews      : {(user_counts < 5).sum():,} ({(user_counts < 5).mean()*100:.1f}%)")
print(f"Users with >= 5 reviews     : {(user_counts >= 5).sum():,} ({(user_counts >= 5).mean()*100:.1f}%)")
print(f"Users with >= 20 reviews    : {(user_counts >= 20).sum():,} ({(user_counts >= 20).mean()*100:.1f}%)")
print(f"\nMedian reviews per user : {user_counts.median()}")
print(f"Mean reviews per user   : {user_counts.mean():.2f}")
print(f"Max reviews per user    : {user_counts.max()}")

Users with exactly 1 review : 175,389 (68.5%)
Users with < 5 reviews      : 232,464 (90.8%)
Users with >= 5 reviews     : 23,592 (9.2%)
Users with >= 20 reviews    : 1,890 (0.7%)

Median reviews per user : 1.0
Mean reviews per user   : 2.22
Max reviews per user    : 448


In [8]:
product_counts = df.groupby('ProductId').size()

print(f"Products with exactly 1 review : {(product_counts == 1).sum():,} ({(product_counts == 1).mean()*100:.1f}%)")
print(f"Products with < 5 reviews      : {(product_counts < 5).sum():,} ({(product_counts < 5).mean()*100:.1f}%)")
print(f"Products with >= 5 reviews     : {(product_counts >= 5).sum():,} ({(product_counts >= 5).mean()*100:.1f}%)")

top1pct = product_counts.quantile(0.99)
top1pct_reviews = product_counts[product_counts >= top1pct].sum()
print(f"\nTop 1% products capture : {top1pct_reviews/len(df)*100:.1f}% of all reviews")
print(f"(popularity bias signal)")

Products with exactly 1 review : 30,408 (40.9%)
Products with < 5 reviews      : 53,843 (72.5%)
Products with >= 5 reviews     : 20,415 (27.5%)

Top 1% products capture : 28.7% of all reviews
(popularity bias signal)


In [9]:
# Filter both users AND products with >= 5 reviews
active_users = user_counts[user_counts >= 5].index
active_products = product_counts[product_counts >= 5].index

df_filtered = df[
    df['UserId'].isin(active_users) & 
    df['ProductId'].isin(active_products)
].copy()

print(f"Before filtering : {len(df):,} reviews | {df['UserId'].nunique():,} users | {df['ProductId'].nunique():,} products")
print(f"After filtering  : {len(df_filtered):,} reviews | {df_filtered['UserId'].nunique():,} users | {df_filtered['ProductId'].nunique():,} products")
print(f"Retained         : {len(df_filtered)/len(df)*100:.1f}% of reviews")

# Recalculate sparsity on filtered set
n_u = df_filtered['UserId'].nunique()
n_p = df_filtered['ProductId'].nunique()
sparsity_filtered = 1 - (len(df_filtered) / (n_u * n_p))
print(f"\nSparsity after filtering : {sparsity_filtered*100:.4f}%")

Before filtering : 568,427 reviews | 256,056 users | 74,258 products
After filtering  : 219,515 reviews | 23,261 users | 17,538 products
Retained         : 38.6% of reviews

Sparsity after filtering : 99.9462%


In [10]:
cutoff = df_filtered['Time'].quantile(0.80)

train = df_filtered[df_filtered['Time'] <= cutoff].copy()
test  = df_filtered[df_filtered['Time'] > cutoff].copy()

# Only keep test interactions where user AND product exist in train
train_users    = set(train['UserId'].unique())
train_products = set(train['ProductId'].unique())
test = test[
    test['UserId'].isin(train_users) & 
    test['ProductId'].isin(train_products)
].copy()

print(f"Cutoff date : {cutoff}")
print(f"Train       : {len(train):,} reviews")
print(f"Test        : {len(test):,} reviews")
print(f"Train users : {train['UserId'].nunique():,}")
print(f"Test users  : {test['UserId'].nunique():,}")

Cutoff date : 2012-04-18 00:00:00
Train       : 175,802 reviews
Test        : 18,083 reviews
Train users : 20,371
Test users  : 4,680


In [11]:
df_filtered.to_csv('df_filtered.csv', index=False)
train.to_csv('train.csv', index=False)
test.to_csv('test.csv', index=False)

print("Saved:")
print(f"  df_filtered.csv : {len(df_filtered):,} rows")
print(f"  train.csv       : {len(train):,} rows")
print(f"  test.csv        : {len(test):,} rows")

Saved:
  df_filtered.csv : 219,515 rows
  train.csv       : 175,802 rows
  test.csv        : 18,083 rows


In [12]:
import os
print(os.getcwd())
print(os.listdir('.'))

/kaggle/working
['train.csv', 'test.csv', '__notebook__.ipynb', 'df_filtered.csv']


In [13]:
print("=" * 50)
print("PHASE 1 COMPLETE — TRACKER NUMBERS")
print("=" * 50)
print(f"Raw dataset          : 568,427 reviews")
print(f"Users (raw)          : 256,056")
print(f"Products (raw)       : 74,258")
print(f"Sparsity (raw)       : 99.9970%")
print(f"Users with 1 review  : 68.5%")
print(f"Users with <5 reviews: 90.8%")
print(f"Top 1% products      : 28.7% of all reviews")
print(f"After 5-review filter: 219,515 reviews")
print(f"Users (filtered)     : 23,261")
print(f"Products (filtered)  : 17,538")
print(f"Sparsity (filtered)  : 99.9462%")
print(f"Train size           : 175,802")
print(f"Test size            : 18,083")
print(f"Train/test cutoff    : 2012-04-18")
print(f"Date range           : 1999-10-08 to 2012-10-26")

PHASE 1 COMPLETE — TRACKER NUMBERS
Raw dataset          : 568,427 reviews
Users (raw)          : 256,056
Products (raw)       : 74,258
Sparsity (raw)       : 99.9970%
Users with 1 review  : 68.5%
Users with <5 reviews: 90.8%
Top 1% products      : 28.7% of all reviews
After 5-review filter: 219,515 reviews
Users (filtered)     : 23,261
Products (filtered)  : 17,538
Sparsity (filtered)  : 99.9462%
Train size           : 175,802
Test size            : 18,083
Train/test cutoff    : 2012-04-18
Date range           : 1999-10-08 to 2012-10-26
